In [1]:
import numpy as np
import pandas as pd
import os

# skLearn - ML Library

# Many models in skLearn
# linear_model is a "sublibrary"
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge # model objects

from sklearn.model_selection import train_test_split # function
from sklearn.preprocessing import StandardScaler, MinMaxScaler # objects
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, precision_score, roc_auc_score
from sklearn.metrics import f1_score, accuracy_score

## 1) 

I'm using the **dataset.csv** dataset. Each observation is a date. The response variable, named `direction_t1`, represents whether return_t1 (The actual percentage return of the stock tomorrow) went up (1) or went down/stayed flat (0). All other variables except `direction_t2`, `direction_t3`, and `return_t1` are the predictors and they represent a number of statistical measures extracted from GDELT and yfinance.

### a)

Read the data as a DataFrame.

In [2]:
df = pd.read_csv('dataset.csv')

df = df.dropna() # losing about 21 rows (the first few days of 2024/2025), 
                    # but necessary to not get error later that says Input X contains NaN. 
                    # LogisticRegression does not accept missing values encoded as NaN natively.

df.head() # dataframe method

,date,ticker,open,high,low,close,volume,daily_return,realized_vol5,article_count,...,polarity_lag2,polarity_lag3,tone_rolling5,tone_momentum,tone_vol5,earnings_week,direction_t1,direction_t2,direction_t3,return_t1
3,2024-12-31,AAPL,251.068493,251.903926,248.074837,249.059464,39480700,-0.007058,0.003577,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.0,0.0,0.0,-0.026236
4,2025-01-02,AAPL,247.577549,247.746638,240.506192,242.525162,55740700,-0.026236,0.008071,232.0,...,0.000000,0.000000,0.164462,0.822311,0.367749,0,0.0,1.0,0.0,-0.002009
5,2025-01-03,AAPL,242.037827,242.853364,240.575812,242.037827,40244100,-0.002009,0.009074,232.0,...,0.000000,0.000000,0.200371,-0.642766,0.356262,0,1.0,0.0,0.0,0.006739
6,2025-01-06,AAPL,242.982661,245.986258,241.878691,243.668915,45045600,0.006739,0.012385,200.0,...,6.450502,0.000000,0.261372,0.125458,0.339073,0,0.0,0.0,0.0,-0.011388
7,2025-01-07,AAPL,241.659894,244.215939,240.038760,240.894089,40856000,-0.011388,0.012227,210.0,...,6.495570,6.450502,0.202943,-0.597149,0.412577,0,1.0,0.0,0.0,0.002023


### b)

Explore the data with Python tools to answer the following questions:

- Is this a binary or a multi-class classification task?
- How many observations and predictors do you have? (The response is not a predictor!)
- Is there any missing data?

In [3]:
# Explore
print(df.info())

# See how many unique values there are in the target variable
print(df['direction_t1'].unique())

# Observations and predictors
print(df.shape)

# Check for missingness
print(df.isna().any(axis = None))

<class 'pandas.core.frame.DataFrame'>
Index: 2233 entries, 3 to 2259
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    2233 non-null   object 
 1   ticker                  2233 non-null   object 
 2   open                    2233 non-null   float64
 3   high                    2233 non-null   float64
 4   low                     2233 non-null   float64
 5   close                   2233 non-null   float64
 6   volume                  2233 non-null   int64  
 7   daily_return            2233 non-null   float64
 8   realized_vol5           2233 non-null   float64
 9   article_count           2233 non-null   float64
 10  unique_source_count     2233 non-null   float64
 11  tone_weighted           2233 non-null   float64
 12  tone_positive_weighted  2233 non-null   float64
 13  tone_negative_weighted  2233 non-null   float64
 14  tone_polarity_weighted  2233 non-null   float

- This is a binary classification task. direction_t1 only has values of 0 and 1. 
- There are 2261 observations and 38 variables. 
- But direction_t2, direction_t3, and return_t1 cannot be used as predictors because they either are the variables I'm trying to predict or reveal future price movement. So there will be 34 predictors and one response.
- There is not any missing data. When I got the data from GDELT and yfinance, I filled missing sentiment rows with 0 (no news = neutral signal). 

### c)

We will be training Jan–Dec 2025 and testing Q1 2026.

In [4]:
# Ensure date column is actually a datetime object
df['date'] = pd.to_datetime(df['date'])

# Define split boundary
split_date = pd.Timestamp('2026-01-01')

# Create the Training and Test sets based on the calendar
train_df = df[df['date'] < split_date]
test_df  = df[df['date'] >= split_date]

# Define y (Target)
y_train = train_df['direction_t1']
y_test  = test_df['direction_t1']

# Define X (Predictors) - Drop the "future" and identifiers
forbidden = ['direction_t1', 'direction_t2', 'direction_t3', 'return_t1', 'date', 'ticker']
X_train = train_df.drop(columns=forbidden)
X_test  = test_df.drop(columns=forbidden)

### d)

(Standard) scale the training and test sets.

In [5]:
# Scale the data
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train) # Scale the training data
X_test_scaled = scaler.transform(X_test) # Scale the test data (do NOT refit)

### e)

Create and train a Logistic Regression model. Do **not** use any Lasso or Ridge penalty.

In [6]:
import time

start = time.time()

# Create a model
model_cl = LogisticRegression(penalty = None) # penalty = None means no regularization

# Train the model
model_cl.fit(X_train_scaled, y_train)

end = time.time()
iteration_runtime = end - start
print(f"Cell took {iteration_runtime:.4f} seconds.")

Cell took 0.1563 seconds.


### f)

Evaluate the model by printing its accuracy, recall and confusion matrix.

In [7]:
# Get the predictions
y_pred = model_cl.predict(X_test_scaled) # The class predictions are obtained with a threshold of 0.5 (default threshold)

# Evaluate
print(accuracy_score(y_test, y_pred)*100)
print(recall_score(y_test, y_pred)*100)
# print(precision_score(y_test, y_pred)*100)
print(confusion_matrix(y_test, y_pred))
print(f1_score(y_test, y_pred, average='macro'))

# Note that y_test is always the first input!!

49.57983193277311
64.13502109704642
[[ 84 155]
 [ 85 152]]
0.4852941176470588


- The accuracy is 49.57983193277311%
- The recall is 64.13502109704642%
- The confusion matrix is 
[[ 84 155]
 [ 85 152]]
- The F1 score is 0.4852941176470588

### g)

Did you use any hyperparameters in the classifier? Was there anything to cross-validate in the classifier? **(15 points)**

**Note:** The decision threshold is not considered a hyperparameter, as it is not a part of the model itself.

We did not use any hyperparameters. Part E specified that we should not use any Lasso or Ridge penalty, so we set penalty = None for the Logistic Regression model. As a result, we did not tune any hyperparameters like penalty or C, and there was nothing to cross-validate. (We only used one model configuration (a single train-test split) so there was nothing to cross-validate for model selection or hyperparameter tuning.) 

In [8]:
# Calculate metrics used in success criteria
val_f1 = f1_score(y_test, y_pred, average='macro')
val_acc = accuracy_score(y_test, y_pred)
val_recall = recall_score(y_test, y_pred)

# Define the metadata for this run
# Manually update 'model_desc' and 'feat_desc' when you change your model!
model_desc = "baseline: LogisticRegression"
feat_desc = "All sentiment + lags + earnings_week"

# 3. Create the log row
log_data = {
    "model": model_desc,
    "features": feat_desc,
    "f1_macro": round(val_f1, 4),
    "accuracy": round(val_acc, 4),
    "recall": round(val_recall, 4),
    "runtime_sec": round(iteration_runtime, 4)
}

# Write to TSV (Tab-Separated Values)
log_df = pd.DataFrame([log_data])
file_path = 'results.tsv'

# If the file doesn't exist, write with header. If it does, append without header.
if not os.path.isfile(file_path):
    log_df.to_csv(file_path, sep='\t', index=False)
else:
    log_df.to_csv(file_path, sep='\t', index=False, mode='a', header=False)

print(f"✓ Results for '{model_desc}' logged to results.tsv")

✓ Results for 'baseline: LogisticRegression' logged to results.tsv
